# Video Clone — Colab remote worker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daokimluc/video-clone-colab/blob/main/video_clone_colab_backend.ipynb)

## Cách dùng
1. **Runtime → Run all**
2. Cell Tunnel in `REMOTE_URL` + `SHARED_SECRET` (chờ 10–40s sau dòng `Starting: cloudflared…`)
3. App **Cấu hình → Remote**: dán cả URL và secret

Nếu port 8765 bận: notebook tự free port. Hoặc **Runtime → Restart session**.

## 1) Cài dependency + cloudflared

In [ ]:
!pip -q install fastapi uvicorn httpx

import shutil

def ensure_cloudflared():
    if shutil.which('cloudflared'):
        print('cloudflared:', shutil.which('cloudflared'))
        return
    print('Installing cloudflared…')
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    !wget -q -O /tmp/cloudflared {url}
    !chmod +x /tmp/cloudflared
    !cp /tmp/cloudflared /usr/local/bin/cloudflared
    print('installed:', shutil.which('cloudflared'))

ensure_cloudflared()
!cloudflared --version

## 2) Shared secret

In [ ]:
import secrets
SHARED_SECRET = secrets.token_urlsafe(16)
print('=' * 60)
print('SHARED_SECRET:')
print(SHARED_SECRET)
print('=' * 60)

## 3) Start FastAPI worker (free port if busy)

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import uvicorn, threading, time, socket, subprocess, httpx

PREFERRED_PORT = 8765

def port_in_use(port: int, host: str = '127.0.0.1') -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def free_port(port: int) -> None:
    if not port_in_use(port):
        print(f'Port {port} free')
        return
    print(f'Port {port} busy — killing…')
    subprocess.run(['bash', '-lc', f'fuser -k {port}/tcp || true'], check=False)
    time.sleep(0.8)
    if port_in_use(port):
        subprocess.run(
            ['bash', '-lc', f'pids=$(lsof -t -i:{port} 2>/dev/null); [ -n "$pids" ] && kill -9 $pids || true'],
            check=False,
        )
        time.sleep(0.8)
    print('Port free' if not port_in_use(port) else 'WARNING still busy')

def pick_port(start: int = PREFERRED_PORT) -> int:
    free_port(start)
    if not port_in_use(start):
        return start
    for p in range(start + 1, start + 20):
        if not port_in_use(p):
            print('Using port', p)
            return p
    raise RuntimeError('No free port')

WORKER_PORT = pick_port()
app = FastAPI(title='VideoClone Colab Worker')

def check(secret: str | None):
    if secret != SHARED_SECRET:
        raise HTTPException(401, 'bad secret')

@app.get('/health')
def health(x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {'ok': True, 'worker': 'colab', 'port': WORKER_PORT}

@app.get('/capabilities')
def caps(x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {'asr': True, 'tts': True}

class AsrReq(BaseModel):
    note: str = 'stub'

@app.post('/infer/asr')
def infer_asr(body: AsrReq, x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {'status': 'not_implemented', 'note': body.note}

def _run():
    uvicorn.run(app, host='127.0.0.1', port=WORKER_PORT, log_level='warning')

threading.Thread(target=_run, daemon=True).start()
time.sleep(1.5)

ok = False
for _ in range(12):
    try:
        r = httpx.get(
            f'http://127.0.0.1:{WORKER_PORT}/health',
            headers={'X-VC-Secret': SHARED_SECRET},
            timeout=3.0,
        )
        print('Local health:', r.status_code, r.json())
        ok = r.status_code == 200
        break
    except Exception as e:
        last = e
        time.sleep(0.4)
if not ok:
    print('Worker failed:', last)
else:
    print(f'Worker OK on http://127.0.0.1:{WORKER_PORT} — next: Tunnel cell')

## 4) Tunnel — chờ URL (10–40 giây)

Sau dòng `Starting: cloudflared…` sẽ có log; khi thấy `https://….trycloudflare.com` là xong.

In [ ]:
import re, subprocess, time, shutil, httpx, sys

assert shutil.which('cloudflared'), 'Install cell first'
assert 'SHARED_SECRET' in globals(), 'Secret cell first'
assert 'WORKER_PORT' in globals(), 'Worker cell first'

# stop old tunnel
if globals().get('_VC_TUNNEL_PROC') is not None:
    try:
        _VC_TUNNEL_PROC.terminate()
        time.sleep(0.6)
    except Exception:
        pass

# kill leftover cloudflared
subprocess.run(['bash', '-lc', 'pkill -f "cloudflared tunnel" || true'], check=False)
time.sleep(0.5)

cmd = [
    'cloudflared', 'tunnel',
    '--url', f'http://127.0.0.1:{WORKER_PORT}',
    '--no-autoupdate',
]
print('Starting:', ' '.join(cmd), flush=True)
print('Waiting for trycloudflare.com URL (up to 90s)…', flush=True)

# unbuffered line output (cloudflared often logs to stderr)
_VC_TUNNEL_PROC = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**dict(**__import__('os').environ), 'PYTHONUNBUFFERED': '1'},
)

REMOTE_URL = None
# accept trycloudflare and cfargotunnel variants
patterns = [
    re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com'),
    re.compile(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com'),
]
deadline = time.time() + 90
lines = []

while time.time() < deadline and REMOTE_URL is None:
    line = _VC_TUNNEL_PROC.stdout.readline()
    if not line:
        if _VC_TUNNEL_PROC.poll() is not None:
            print('cloudflared exited early code=', _VC_TUNNEL_PROC.returncode, flush=True)
            break
        time.sleep(0.05)
        continue
    line = line.rstrip()
    lines.append(line)
    # show progress so cell doesn't look stuck
    print(line, flush=True)
    for pat in patterns:
        m = pat.search(line)
        if m:
            REMOTE_URL = m.group(0).rstrip('/')
            break

if not REMOTE_URL:
    print('\n--- failed to parse URL; last lines ---', flush=True)
    print('\n'.join(lines[-50:]), flush=True)
    raise RuntimeError(
        'Chưa có REMOTE_URL. Thử Runtime→Restart session, Run all. '
        'Hoặc kiểm tra mạng Colab / cloudflared.'
    )

print('\n' + '=' * 60, flush=True)
print('✅ COPY VÀO APP DESKTOP:', flush=True)
print('REMOTE_URL =', REMOTE_URL, flush=True)
print('SHARED_SECRET =', SHARED_SECRET, flush=True)
print('Backend mode = Remote', flush=True)
print('=' * 60, flush=True)

time.sleep(2)
try:
    hr = httpx.get(
        REMOTE_URL + '/health',
        headers={'X-VC-Secret': SHARED_SECRET},
        timeout=25.0,
        follow_redirects=True,
    )
    print('Public /health:', hr.status_code, hr.text[:200], flush=True)
except Exception as e:
    print('Public probe (retry later):', e, flush=True)

print('Giữ cell/runtime chạy. Gửi REMOTE_URL + SHARED_SECRET cho app / agent test.')

### Sau dòng `Starting: cloudflared…`

- **Bình thường:** 10–40s, log cuộn, rồi xuất hiện `https://xxxx.trycloudflare.com`
- **Kẹt > 90s không URL:** Restart session → Run all
- **Cần gửi cho desktop:** cả `REMOTE_URL` **và** `SHARED_SECRET`